In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import brentq
import matplotlib.pyplot as plt

In [ ]:
# ================================================================
# HELPER FUNCTIONS
# ================================================================

def norm_cdf(x): return norm.cdf(x)
def norm_ppf(x): return norm.ppf(x)

LGD = 0.60        # ISDA standard for sovereign CDS
T   = 5.0         # 5-year horizon (matches 5Y CDS)


def pd_to_cds_spread(pd_val, T=T, lgd=LGD):
    """PD → approximate CDS spread in bps."""
    pd_c = np.clip(pd_val, 1e-12, 1 - 1e-6)
    return -(1.0 / T) * np.log(1.0 - lgd * pd_c) * 10000


def merton_pd_cf(V0, B0, sigma, mu, T=T):
    """Closed-form Merton PD and DD."""
    denom = np.maximum(sigma * np.sqrt(T), 1e-10)
    dd = (np.log(V0 / B0) + (mu - 0.5 * sigma**2) * T) / denom
    pd = norm.cdf(-dd)
    return pd, dd


# ================================================================
# ROLLING CALIBRATION
# ================================================================

def _calibrate_window(B0, sigma, mu, cds_obs, T=T, lgd=LGD,
                      scale_bounds=(1.01, 10.0)):
    """
    Find scale_mult for a window so M1's avg implied spread
    matches the avg observed CDS spread.
    Returns (scale, status_str).
    """
    valid = (~np.isnan(B0) & ~np.isnan(sigma) & ~np.isnan(mu) &
             ~np.isnan(cds_obs) & (B0 > 0) & (sigma > 0) & (cds_obs > 0))
    B0_v, sig_v, mu_v, cds_v = B0[valid], sigma[valid], mu[valid], cds_obs[valid]

    if len(B0_v) < 10:
        return None, 'insufficient_data'

    target = np.mean(cds_v)

    def err(s):
        pd_vals, _ = merton_pd_cf(s * B0_v, B0_v, sig_v, mu_v, T)
        return np.mean(pd_to_cds_spread(pd_vals, T, lgd)) - target

    try:
        e_lo, e_hi = err(scale_bounds[0]), err(scale_bounds[1])
    except Exception:
        return None, 'eval_failed'

    if e_lo < 0:
        return scale_bounds[0], 'at_lower_bound'
    if e_hi > 0:
        return scale_bounds[1], 'at_upper_bound'

    try:
        return brentq(err, scale_bounds[0], scale_bounds[1],
                      xtol=1e-4, maxiter=100), 'converged'
    except Exception:
        return None, 'brentq_failed'


def calibrate_rolling(df, cds_col='cds_spread',
                      barrier_col='default_barrier',
                      sigma_col='msci_vol_annual',
                      country_col='country_clean',
                      mu_mode='zero',
                      window_weeks=52,
                      min_obs=26,
                      T=T, lgd=LGD,
                      scale_bounds=(1.01, 10.0)):
    """
    Rolling calibration: at each week t, find scale_mult using
    the trailing window so M1's avg implied spread ≈ avg CDS.
    
    Handles structural breaks naturally.
    """
    df_out = df.copy()
    df_out['scale_mult'] = np.nan

    print(f'Rolling calibration: window={window_weeks}w, '
          f'min_obs={min_obs}, LGD={lgd}')

    for country in sorted(df_out[country_col].unique()):
        mask = df_out[country_col] == country
        idx = df_out.loc[mask].index

        B0_all  = df_out.loc[mask, barrier_col].values
        sig_all = df_out.loc[mask, sigma_col].values
        cds_all = df_out.loc[mask, cds_col].values
        mu_all  = np.zeros(len(idx))

        n = len(idx)
        last_good = None
        conv = 0

        for t in range(n):
            start = max(0, t - window_weeks + 1)
            if t - start + 1 < min_obs:
                if last_good is not None:
                    df_out.loc[idx[t], 'scale_mult'] = last_good
                continue

            scale, status = _calibrate_window(
                B0_all[start:t+1], sig_all[start:t+1],
                mu_all[start:t+1], cds_all[start:t+1],
                T, lgd, scale_bounds)

            if scale is not None:
                df_out.loc[idx[t], 'scale_mult'] = scale
                last_good = scale
                if status == 'converged': conv += 1
            elif last_good is not None:
                df_out.loc[idx[t], 'scale_mult'] = last_good

        # Fill edges
        s = df_out.loc[mask, 'scale_mult']
        df_out.loc[mask, 'scale_mult'] = s.ffill().bfill()

        scales = df_out.loc[mask, 'scale_mult'].dropna()
        print(f'  {country:<20s}  scale: {scales.mean():.2f} '
              f'[{scales.min():.2f}–{scales.max():.2f}]  conv={conv}/{n}')

    return df_out

# ================================================================
# ROLLING CALIBRATION FOR JUMP MODELS (MC-based)
# ================================================================

def _calibrate_window_mc(B0, sigma, mu, cds_obs, lam, theta, delta,
                         T=5.0, lgd=0.6, scale_bounds=(1.01, 10.0),
                         n_paths_cal=5000, seed=99):
    """Find scale_mult for a window so MC-model avg spread ≈ avg CDS."""
    valid = (~np.isnan(B0) & ~np.isnan(sigma) & ~np.isnan(cds_obs)
             & (B0 > 0) & (sigma > 0) & (cds_obs > 0))
    B0_v = B0[valid]; sig_v = sigma[valid]; mu_v = mu[valid]
    cds_v = cds_obs[valid]; lam_v = lam[valid]
    th_v = theta[valid]; de_v = delta[valid]

    if len(B0_v) < 10:
        return None, 'insufficient_data'

    target = np.mean(cds_v)

    def err(s):
        V0_try = s * B0_v
        pd_vals, _, _ = mc_pd_dd_jump_stochastic(
            V0_try, B0_v, sig_v, mu_v, lam_v, th_v, de_v,
            T=T, n_paths=n_paths_cal, seed=seed)
        return np.mean(pd_to_cds_spread(pd_vals, T, lgd)) - target

    try:
        e_lo, e_hi = err(scale_bounds[0]), err(scale_bounds[1])
    except Exception:
        return None, 'eval_failed'
    if e_lo < 0: return scale_bounds[0], 'at_lower_bound'
    if e_hi > 0: return scale_bounds[1], 'at_upper_bound'
    try:
        return brentq(err, scale_bounds[0], scale_bounds[1],
                      xtol=1e-3, maxiter=50), 'converged'
    except Exception:
        return None, 'brentq_failed'


def calibrate_rolling_mc(df, lam_col, theta_col='theta', delta_col='delta',
                         cds_col='cds_spread',
                         barrier_col='default_barrier',
                         sigma_col='msci_vol_annual',
                         country_col='country_clean',
                         window_weeks=52, min_obs=26,
                         n_paths_cal=5000, seed=99,
                         scale_col_out='scale_mult_mc'):
    """Rolling calibration for a jump model (MC-based)."""
    df[scale_col_out] = np.nan
    for country in sorted(df[country_col].unique()):
        mask = df[country_col] == country
        idx = df.loc[mask].index
        B0_a  = df.loc[mask, barrier_col].values
        sig_a = df.loc[mask, sigma_col].values
        cds_a = df.loc[mask, cds_col].values
        mu_a  = np.zeros(len(idx))
        lam_a = df.loc[mask, lam_col].values
        th_a  = df.loc[mask, theta_col].values
        de_a  = df.loc[mask, delta_col].values
        n = len(idx); last_good = None; conv = 0
        for t in range(n):
            start = max(0, t - window_weeks + 1)
            if t - start + 1 < min_obs:
                if last_good is not None:
                    df.loc[idx[t], scale_col_out] = last_good
                continue
            sl = slice(start, t+1)
            scale, status = _calibrate_window_mc(
                B0_a[sl], sig_a[sl], mu_a[sl], cds_a[sl],
                lam_a[sl], th_a[sl], de_a[sl],
                n_paths_cal=n_paths_cal, seed=seed)
            if scale is not None:
                df.loc[idx[t], scale_col_out] = scale
                last_good = scale
                if status == 'converged': conv += 1
            elif last_good is not None:
                df.loc[idx[t], scale_col_out] = last_good
        s = df.loc[mask, scale_col_out]
        df.loc[mask, scale_col_out] = s.ffill().bfill()
        scales = df.loc[mask, scale_col_out].dropna()
        print(f'  {country:<20s}  {scale_col_out}: {scales.mean():.2f} '
              f'[{scales.min():.2f}-{scales.max():.2f}]  conv={conv}/{n}')
    return df


In [ ]:
# ================================================================
# TWO-PHASE MONTE CARLO
# ================================================================
#
# Phase 1 (0 to T_jump weeks):
#   Jump-diffusion with lambda from OVX regime.
#   OVX is forward-looking ~30 days, so T_jump = 4/52 years.
#   The asset can experience discrete negative shocks here.
#
# Phase 2 (T_jump to T):
#   Pure GBM (smooth diffusion only).
#   No jumps — we don't know future oil stress beyond 1 month.
#
# This is more honest than applying lambda over the full 5-year
# horizon. The jump component only affects the near-term path,
# which is where the OVX signal actually has information.
# ================================================================

def mc_two_phase(V0, B0, sigma, mu, lam, theta, delta,
                 T=5.0, T_jump=4/52, n_paths=20000, seed=7):
    """
    Two-phase Monte Carlo:
      Phase 1 [0, T_jump]: jump-diffusion (lam, theta, delta)
      Phase 2 [T_jump, T]: pure GBM
    
    Parameters
    ----------
    lam:    jump intensity for Phase 1 (vector, per year)
    theta:  mean log-jump size, < 0 (vector)
    delta:  std of log-jump size (vector)
    T_jump: duration of jump phase in years (default 4/52)
    """
    rng = np.random.default_rng(seed)
    n_obs = len(V0)
    T_diff = T - T_jump   # Phase 2 duration

    # ---- Phase 1: Jump-diffusion over T_jump ----
    drift_1 = (mu - 0.5 * sigma**2) * T_jump
    vol_1   = sigma * np.sqrt(T_jump)

    # Diffusion in Phase 1
    Z1 = rng.standard_normal((n_obs, n_paths))

    # Jumps in Phase 1
    N = rng.poisson(lam[:, None] * T_jump, size=(n_obs, n_paths))
    agg_mean = N * theta[:, None]
    agg_std  = np.sqrt(np.maximum(N, 0)) * delta[:, None]
    Z_jump   = rng.standard_normal((n_obs, n_paths))
    total_jump = np.minimum(agg_mean + agg_std * Z_jump, 0.0)

    # Log asset value at end of Phase 1
    logV_phase1 = (np.log(V0)[:, None]
                   + drift_1[:, None]
                   + vol_1[:, None] * Z1
                   + total_jump)

    # ---- Phase 2: Pure GBM over T_diff ----
    drift_2 = (mu - 0.5 * sigma**2) * T_diff
    vol_2   = sigma * np.sqrt(T_diff)

    Z2 = rng.standard_normal((n_obs, n_paths))

    logVT = logV_phase1 + drift_2[:, None] + vol_2[:, None] * Z2

    # ---- Metrics ----
    PD = (logVT < np.log(B0)[:, None]).mean(axis=1)
    DD = norm_ppf(1 - np.clip(PD, 1e-10, 1 - 1e-10))
    SE = np.sqrt(PD * (1 - PD) / n_paths)

    return PD, DD, SE


# ================================================================
# MAIN RUNNER
# ================================================================

def run_all_models(panel_csv, out_csv,
                   mu_mode='zero',
                   window_weeks=52,
                   ovx_thresholds=(30, 50),
                   ovx_lambdas=(0.01, 0.05, 0.20),
                   T_jump_weeks=4,
                   n_paths=50000):
    """
    Pipeline:
      1. Load panel, classify countries
      2. Rolling-calibrate V0/B0 (M1 -> trailing CDS)
      3. Run M1 (closed-form GBM)
      4. Run M2 (two-phase: constant lambda for Phase 1, GBM for Phase 2)
      5. Run M3 (two-phase: OVX lambda for Phase 1, GBM for Phase 2)
    """
    df = pd.read_csv(panel_csv, parse_dates=['date'])

    # --- Oil exporter classification ---
    OIL_EXPORTERS = ['Saudi Arabia', 'United Arab Emirates', 'Qatar',
                     'Colombia', 'Mexico', 'Brazil', 'Egypt', 'Malaysia']
    CONTROLS      = ['Indonesia', 'Philippines', 'Turkey', 'Chile',
                     'China', 'South Africa', 'South Korea', 'Thailand']
    _oil_set  = {c.lower() for c in OIL_EXPORTERS}
    _ctrl_set = {c.lower() for c in CONTROLS}
    df['oil_exporter'] = df['country_clean'].str.lower().apply(
        lambda x: 1 if x in _oil_set else (0 if x in _ctrl_set else np.nan))
    n_oil  = (df['oil_exporter'] == 1).sum()
    n_ctrl = (df['oil_exporter'] == 0).sum()
    n_oth  = df['oil_exporter'].isna().sum()
    print(f'  Groups: {n_oil} oil-exporter rows, {n_ctrl} control rows, {n_oth} other')
    if n_oth > 0:
        unmatched = df.loc[df['oil_exporter'].isna(), 'country_clean'].unique()
        print(f'  Unmatched countries: {list(unmatched)}')

    # --- Drift ---
    if mu_mode == 'hist':
        df['mu_hist'] = (
            df.groupby('country_clean')['msci_ret_weekly']
            .transform(lambda x: x.rolling(52, min_periods=10)
                       .mean().shift(1) * 52)
            .fillna(0)
        )

    # --- Rolling calibration of V0/B0 (on M1) ---
    print('=' * 60)
    print('  STEP 1: Rolling Calibration (M1 -> observed CDS)')
    print('=' * 60)
    df = calibrate_rolling(
        df, cds_col='cds_spread',
        window_weeks=window_weeks,
    )

    n_before = len(df)
    df = df.dropna(subset=['scale_mult']).copy()
    print(f'\n  Kept {len(df)}/{n_before} rows with valid scale')

    # --- Build arrays ---
    sigma = df['msci_vol_annual'].values
    B0    = df['default_barrier'].values
    V0    = df['scale_mult'].values * B0
    theta = df['theta'].values
    delta = df['delta'].values
    ovx   = df['OVX'].values
    mu    = df['mu_hist'].values if mu_mode == 'hist' else np.zeros(len(df))

    T_jump = T_jump_weeks / 52.0
    print(f'\n  T_jump = {T_jump_weeks} weeks ({T_jump:.4f} years)')
    print(f'  Phase 1 (jump-diffusion): 0 to {T_jump:.3f}y')
    print(f'  Phase 2 (pure GBM):       {T_jump:.3f}y to {T:.1f}y')

    # ---- M1: Baseline Merton (closed-form, full GBM) ----
    print('\n' + '=' * 60)
    print('  STEP 2: Running Models')
    print('=' * 60)

    pd_m1, dd_m1 = merton_pd_cf(V0, B0, sigma, mu)

    # ---- Lambda arrays ----
    low_t, high_t = ovx_thresholds
    lam_calm, lam_elev, lam_stress = ovx_lambdas

    lam_ovx = np.where(
        ovx < low_t,  lam_calm,
        np.where(ovx < high_t, lam_elev, lam_stress)
    )
    df['lam_ovx'] = lam_ovx

    lambda_const = np.mean(lam_ovx)
    print(f'  M2 lam_const (from M3 avg): {lambda_const:.4f}')
    lam_const_arr = np.full(len(df), lambda_const)

    # ---- M2: Two-phase with constant lambda ----
    print('  Running M2 (two-phase, constant lam)...')
    pd_m2, dd_m2, _ = mc_two_phase(
        V0, B0, sigma, mu, lam_const_arr, theta, delta,
        T=T, T_jump=T_jump, n_paths=n_paths, seed=11)

    # ---- M3: Two-phase with OVX lambda ----
    print('  Running M3 (two-phase, OVX lam)...')
    pd_m3, dd_m3, _ = mc_two_phase(
        V0, B0, sigma, mu, lam_ovx, theta, delta,
        T=T, T_jump=T_jump, n_paths=n_paths, seed=22)

    # ---- Store ----
    df['pd_cf_gbm']        = pd_m1;  df['dd_cf_gbm']        = dd_m1
    df['pd_mc_jump_const'] = pd_m2;  df['dd_mc_jump_const'] = dd_m2
    df['pd_mc_jump_ovx']   = pd_m3;  df['dd_mc_jump_ovx']   = dd_m3

    df['spread_m1'] = pd_to_cds_spread(pd_m1)
    df['spread_m2'] = pd_to_cds_spread(pd_m2)
    df['spread_m3'] = pd_to_cds_spread(pd_m3)

    df['dd_diff_m3_m1'] = dd_m3 - dd_m1
    df['dd_diff_m3_m2'] = dd_m3 - dd_m2
    df['dd_diff_m2_m1'] = dd_m2 - dd_m1

    # ---- Summary ----
    print(f'\n{"="*60}')
    print('  MODEL SUMMARY')
    print('=' * 60)
    obs = df['cds_spread']
    for label, col in [('M1 (GBM)', 'spread_m1'),
                       ('M2 (const lam)', 'spread_m2'),
                       ('M3 (OVX lam)',  'spread_m3')]:
        impl = df[col]
        corr = obs.corr(impl)
        rmse = np.sqrt(np.mean((obs - impl)**2))
        print(f'  {label:<15s}  mean={impl.mean():7.0f}bps  '
              f'corr={corr:.3f}  rmse={rmse:.0f}bps')
    print(f'  {"Observed":<15s}  mean={obs.mean():7.0f}bps')

    # DD summary
    print(f'\n  DD Summary (mean):')
    print(f'    M1: {dd_m1.mean():.3f}   M2: {dd_m2.mean():.3f}   M3: {dd_m3.mean():.3f}')
    print(f'    DD gap M3-M1: {(dd_m3 - dd_m1).mean():+.4f}')
    print(f'    DD gap M2-M1: {(dd_m2 - dd_m1).mean():+.4f}')

    df.to_csv(out_csv, index=False)
    print(f'\n  Saved to {out_csv}')
    return df


In [ ]:
in_path  = 'data/processed/structural_model_panel_built.csv'
out_path = 'output/mc_calibrated_results.csv'

df_res = run_all_models(
    panel_csv=in_path,
    out_csv=out_path,
    mu_mode='zero',
    window_weeks=52,
    ovx_thresholds=(30, 50),
    ovx_lambdas=(0.3, 1.1, 2.9),   # calibrated for 4-week window:
                                     #   calm:     P(jump) ~2%
                                     #   elevated: P(jump) ~8%
                                     #   stress:   P(jump) ~20%
    T_jump_weeks=4,                  # OVX is forward-looking ~30 days
    n_paths=10000,                   # increase to 50000 for final
)

In [ ]:
country = 'brazil'   # <-- change here

df_c = (df_res[df_res['country_clean'].str.lower() == country.lower()]
        .sort_values('date').copy())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# --- Top panel: Distance to Default ---
ax = axes[0]
ax.plot(df_c['date'], df_c['dd_cf_gbm'],        label='M1: GBM (closed-form)', lw=2)
ax.plot(df_c['date'], df_c['dd_mc_jump_const'],  label='M2: Jump (constant λ)', alpha=0.8)
ax.plot(df_c['date'], df_c['dd_mc_jump_ovx'],    label='M3: Jump (OVX λ)',      alpha=0.8)
ax.axhline(0, color='black', lw=1, ls=':')
ax.set_ylabel('Distance to Default')
ax.set_title(f'Distance to Default — {country.title()}')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Bottom panel: Implied spread vs observed CDS ---
ax = axes[1]
ax.plot(df_c['date'], df_c['cds_spread'], label='Observed CDS', color='black', lw=2)
ax.plot(df_c['date'], df_c['spread_m1'],  label='M1: GBM',          alpha=0.8)
ax.plot(df_c['date'], df_c['spread_m2'],  label='M2: Jump (const)', alpha=0.8)
ax.plot(df_c['date'], df_c['spread_m3'],  label='M3: Jump (OVX)',   alpha=0.8)
ax.set_ylabel('CDS Spread (bps)')
ax.set_xlabel('Date')
ax.set_title(f'Implied vs Observed CDS Spread — {country.title()}')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Calibration diagnostic: how does scale_mult evolve?
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax = axes[0]
ax.plot(df_c['date'], df_c['scale_mult'], color='tab:purple', lw=2)
ax.set_ylabel('Calibrated Scale (V0 / B0)')
ax.set_title(f'Rolling Scale Calibration — {country.title()}')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(df_c['date'], df_c['cds_spread'], label='Observed CDS', color='black', lw=1.5)
ax2 = ax.twinx()
ax2.plot(df_c['date'], df_c['OVX'], label='OVX', color='tab:red', alpha=0.5)
ax.set_ylabel('CDS Spread (bps)')
ax2.set_ylabel('OVX', color='tab:red')
ax.set_title(f'CDS Spread & OVX — {country.title()}')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import statsmodels.api as sm

def run_reg(x, y):
    X = sm.add_constant(x)
    model = sm.OLS(y, X, missing='drop').fit(cov_type='HC1')
    return model.rsquared, model.params

obs_cds = df_c['cds_spread']

# Compare implied spreads directly against observed CDS
r2_m1, _ = run_reg(df_c['spread_m1'], obs_cds)
r2_m2, _ = run_reg(df_c['spread_m2'], obs_cds)
r2_m3, _ = run_reg(df_c['spread_m3'], obs_cds)

print(f'R² Comparison — {country.title()}')
print(f'  M1 (GBM):           {r2_m1:.4f}')
print(f'  M2 (Jump const λ):  {r2_m2:.4f}')
print(f'  M3 (Jump OVX λ):    {r2_m3:.4f}')

# RMSE
for label, col in [('M1', 'spread_m1'), ('M2', 'spread_m2'), ('M3', 'spread_m3')]:
    rmse = np.sqrt(np.mean((obs_cds - df_c[col])**2))
    corr = obs_cds.corr(df_c[col])
    print(f'  {label}  RMSE={rmse:.0f}bps  Corr={corr:.3f}')

In [ ]:
# ================================================================
# CHANGE-ON-CHANGE REGRESSIONS
# ================================================================
# Why changes? The *level* of implied spread depends on the
# calibrated scale (V0/B0), which is a free parameter.
# But *changes* in implied spread reflect model dynamics:
# does the model move in the right direction, at the right time,
# by the right amount? This is what we actually want to test.
#
# ΔS_obs(t) = α + β × ΔS_model(t) + ε(t)
#
# β ≈ 1 means model changes track observed changes 1:1
# R² tells us how much of the *variation* in CDS moves is explained
# ================================================================

import statsmodels.api as sm


def change_on_change_reg(df_country, cds_col='cds_spread',
                         model_cols=None, country_label=''):
    """
    Regress weekly changes in observed CDS on weekly changes
    in model-implied CDS. Reports R², β, and Newey-West t-stats.
    """
    if model_cols is None:
        model_cols = {
            'M1 (GBM)':       'spread_m1',
            'M2 (const λ)':   'spread_m2',
            'M3 (OVX λ)':     'spread_m3',
        }

    dc = df_country.sort_values('date').copy()
    dc['d_cds'] = dc[cds_col].diff()

    results = []
    for label, col in model_cols.items():
        dc[f'd_{col}'] = dc[col].diff()

        tmp = dc[['d_cds', f'd_{col}']].dropna()
        if len(tmp) < 20:
            continue

        y = tmp['d_cds'].values
        X = sm.add_constant(tmp[f'd_{col}'].values)

        # Newey-West HAC standard errors (4 lags for weekly data)
        reg = sm.OLS(y, X).fit(cov_type='HAC',
                              cov_kwds={'maxlags': 4})

        results.append({
            'model': label,
            'R2': reg.rsquared,
            'beta': reg.params[1],
            't_stat': reg.tvalues[1],
            'p_value': reg.pvalues[1],
            'alpha': reg.params[0],
            'n_obs': int(reg.nobs),
        })

    res_df = pd.DataFrame(results)

    if country_label:
        print(f'\nΔCDS ~ ΔModel Spread — {country_label}')
    print('-' * 70)
    print(f'{"Model":<16s} {"R²":>8s} {"β":>8s} {"t-stat":>8s} '
          f'{"p-val":>8s} {"α":>8s} {"N":>6s}')
    print('-' * 70)
    for _, r in res_df.iterrows():
        sig = '***' if r['p_value'] < 0.01 else \
              '**'  if r['p_value'] < 0.05 else \
              '*'   if r['p_value'] < 0.10 else ''
        print(f'{r["model"]:<16s} {r["R2"]:8.4f} {r["beta"]:8.3f} '
              f'{r["t_stat"]:8.2f}{sig:<3s} '
              f'{r["p_value"]:8.4f} {r["alpha"]:8.2f} {r["n_obs"]:6d}')
    print('-' * 70)

    return res_df


# --- Run for the selected country ---
res_country = change_on_change_reg(df_c, country_label=country.title())

In [ ]:
# ================================================================
# PANEL: Change-on-Change R² by Country + DiD Test
# ================================================================
# For each country: regress ΔCDS on ΔModel_spread, store R².
# Then test: is R² improvement from M1→M3 larger for oil exporters?
# ================================================================

panel_rows = []
for ctry, grp in df_res.groupby('country_clean'):
    oil = grp['oil_exporter'].iloc[0] if 'oil_exporter' in grp.columns else np.nan

    grp = grp.sort_values('date').copy()
    grp['d_cds'] = grp['cds_spread'].diff()

    row = {'country': ctry, 'oil_exporter': oil}

    for m, col in [('m1','spread_m1'),('m2','spread_m2'),('m3','spread_m3')]:
        grp[f'd_{col}'] = grp[col].diff()
        tmp = grp[['d_cds', f'd_{col}']].dropna()
        if len(tmp) < 30:
            continue
        y = tmp['d_cds'].values
        X = sm.add_constant(tmp[f'd_{col}'].values)
        reg = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
        row[f'R2_{m}']   = reg.rsquared
        row[f'beta_{m}'] = reg.params[1]

    panel_rows.append(row)

r2_panel = pd.DataFrame(panel_rows)

# Improvements
r2_panel['R2_gain_m3_vs_m1'] = r2_panel['R2_m3'] - r2_panel['R2_m1']
r2_panel['R2_gain_m3_vs_m2'] = r2_panel['R2_m3'] - r2_panel['R2_m2']

# Display
r2_sorted = r2_panel.sort_values('oil_exporter', ascending=False)
print('Change-on-Change R² by Country')
print('=' * 85)
print(f'{"Country":<20s} {"Oil":>3s} {"R²_M1":>7s} {"R²_M2":>7s} '
      f'{"R²_M3":>7s} {"Δ(M3-M1)":>9s} {"Δ(M3-M2)":>9s}')
print('-' * 85)
for _, r in r2_sorted.iterrows():
    print(f'{r["country"]:<20s} {int(r["oil_exporter"]):>3d} '
          f'{r.get("R2_m1",np.nan):>7.4f} '
          f'{r.get("R2_m2",np.nan):>7.4f} '
          f'{r.get("R2_m3",np.nan):>7.4f} '
          f'{r.get("R2_gain_m3_vs_m1",np.nan):>+9.4f} '
          f'{r.get("R2_gain_m3_vs_m2",np.nan):>+9.4f}')
print('-' * 85)

# Group averages
print('\nGroup Averages:')
for grp_val, label in [(1, 'Oil Exporters'), (0, 'Controls')]:
    sub = r2_panel[r2_panel['oil_exporter'] == grp_val]
    if len(sub) == 0: continue
    print(f'  {label} (n={len(sub)}):')
    print(f'    R²_M1={sub["R2_m1"].mean():.4f}  '
          f'R²_M2={sub["R2_m2"].mean():.4f}  '
          f'R²_M3={sub["R2_m3"].mean():.4f}')
    print(f'    Gain M3 vs M1: {sub["R2_gain_m3_vs_m1"].mean():+.4f}  '
          f'Gain M3 vs M2: {sub["R2_gain_m3_vs_m2"].mean():+.4f}')

# --- DiD test on R² improvement ---
print('\n' + '=' * 60)
print('DiD Test: Is R² improvement larger for oil exporters?')
print('=' * 60)

for dep_var, label in [('R2_gain_m3_vs_m1', 'M3 vs M1 (oil jumps vs baseline)'),
                        ('R2_gain_m3_vs_m2', 'M3 vs M2 (oil calib vs generic)')]:
    valid = r2_panel.dropna(subset=[dep_var])
    y = valid[dep_var].values
    X = sm.add_constant(valid['oil_exporter'].astype(float).values)
    reg = sm.OLS(y, X).fit(cov_type='HC1')

    beta = reg.params[1]
    t = reg.tvalues[1]
    p = reg.pvalues[1]
    sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''

    print(f'\n  {label}:')
    print(f'    β(oil_exporter) = {beta:+.4f}  t={t:.2f}  p={p:.4f} {sig}')
    print(f'    Interpretation: oil exporters gain {beta:+.4f} more R² from M3')

In [ ]:
# ================================================================
# SCATTER: R² improvement vs oil dependence
# ================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in [
    (axes[0], 'R2_gain_m3_vs_m1', 'R² Gain: M3 vs M1 (baseline)'),
    (axes[1], 'R2_gain_m3_vs_m2', 'R² Gain: M3 vs M2 (generic jumps)'),
]:
    valid = r2_panel.dropna(subset=[col])
    oil = valid[valid['oil_exporter'] == 1]
    ctrl = valid[valid['oil_exporter'] == 0]

    ax.scatter(ctrl.index, ctrl[col], color='steelblue',
               label='Controls', alpha=0.7, s=60)
    ax.scatter(oil.index, oil[col], color='tomato',
               label='Oil Exporters', alpha=0.7, s=60, marker='D')

    # Add country labels
    for _, r in valid.iterrows():
        ax.annotate(r['country'][:8], (_, r[col]),
                    fontsize=7, alpha=0.6,
                    xytext=(3, 3), textcoords='offset points')

    ax.axhline(0, color='black', lw=0.8, ls='--')
    ax.set_title(title)
    ax.set_ylabel('R² improvement')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# DD-BASED COMPARISON: M3 vs M1
# ================================================================
#
# Because V0 is calibrated to M1, spread levels are biased for
# M2/M3 (negative-only jumps always raise PD above M1).
# But DD differences are meaningful:
#
#   dd_diff = DD_M3 - DD_M1
#   negative => M3 sees sovereign closer to distress
#
# The test: does this DD gap covary with OBSERVED CDS changes?
# If M3's extra risk signal is real, weeks where M3 pushes DD
# down (relative to M1) should coincide with CDS widening.
#
# Regression:   delta_CDS(t) = a + b * dd_diff_M3_M1(t) + e
#   b < 0 => when M3 sees more risk than M1, CDS widens (good)
#   b ≈ 0 => M3's jump component adds noise, not signal
# ================================================================

import statsmodels.api as sm

# Compute weekly changes
df_res_sorted = df_res.sort_values(['country_clean', 'date'])
df_res_sorted['d_cds']     = df_res_sorted.groupby('country_clean')['cds_spread'].diff()
df_res_sorted['d_spread_m1'] = df_res_sorted.groupby('country_clean')['spread_m1'].diff()
df_res_sorted['d_spread_m3'] = df_res_sorted.groupby('country_clean')['spread_m3'].diff()

# ---- Per-country: does dd_diff predict CDS changes? ----
print('=' * 75)
print('  DD GAP (M3-M1) AS PREDICTOR OF CDS CHANGES')
print('  dCDS(t) = a + b * [DD_M3(t) - DD_M1(t)] + e')
print('  b < 0 and significant => M3 captures risk that M1 misses')
print('=' * 75)

results_rows = []
for ctry in sorted(df_res_sorted['country_clean'].unique()):
    dc = df_res_sorted[df_res_sorted['country_clean'] == ctry].dropna(
        subset=['d_cds', 'dd_diff_m3_m1'])
    if len(dc) < 50:
        continue

    y = dc['d_cds'].values
    X = sm.add_constant(dc['dd_diff_m3_m1'].values)
    try:
        reg = sm.OLS(y, X).fit(cov_type='HAC',
                               cov_kwds={'maxlags': 4})
    except Exception:
        continue

    oil = dc['oil_exporter'].iloc[0]
    results_rows.append({
        'country': ctry, 'oil_exporter': oil,
        'beta': reg.params[1], 'tstat': reg.tvalues[1],
        'pval': reg.pvalues[1], 'R2': reg.rsquared, 'N': int(reg.nobs)
    })

    sig = '***' if reg.pvalues[1] < 0.01 else '**' if reg.pvalues[1] < 0.05 else '*' if reg.pvalues[1] < 0.10 else ''
    tag = 'OIL' if oil == 1 else 'CTL'
    print(f'  {ctry:<20s} [{tag}]  beta={reg.params[1]:>+8.2f}  '
          f't={reg.tvalues[1]:>6.2f}  p={reg.pvalues[1]:.3f}{sig:>4s}  '
          f'R2={reg.rsquared:.4f}  N={int(reg.nobs)}')

dd_results = pd.DataFrame(results_rows)

# ---- Group summary ----
print('\n' + '-' * 75)
for grp, label in [(1, 'Oil Exporters'), (0, 'Controls')]:
    sub = dd_results[dd_results['oil_exporter'] == grp]
    if len(sub) == 0: continue
    pct_sig = (sub['pval'] < 0.10).mean() * 100
    print(f'  {label} (n={len(sub)}):  '
          f'avg beta={sub["beta"].mean():+.2f}  '
          f'avg R2={sub["R2"].mean():.4f}  '
          f'significant at 10%: {pct_sig:.0f}%')

print('\n  KEY: beta < 0 means the DD gap has predictive content.')
print('  Stronger (more negative) beta for oil exporters = M3 adds')
print('  oil-specific risk information that M1 cannot capture.')


In [ ]:
# ================================================================
# CHANGE-ON-CHANGE: dCDS vs dSpread for M1 and M3
# ================================================================
#
# This is calibration-free: we compare CHANGES in model-implied
# spread vs CHANGES in observed CDS. If M3 tracks weekly moves
# better than M1, its beta should be closer to 1 and its R² higher.
#
#   dCDS(t) = a + b * dSpread_model(t) + e
# ================================================================

print('=' * 80)
print('  CHANGE-ON-CHANGE: dCDS = a + b * dSpread_model')
print('  Higher R2 = model tracks weekly CDS moves better')
print('=' * 80)

cc_rows = []
for ctry in sorted(df_res_sorted['country_clean'].unique()):
    dc = df_res_sorted[df_res_sorted['country_clean'] == ctry].dropna(
        subset=['d_cds', 'd_spread_m1', 'd_spread_m3'])
    if len(dc) < 50:
        continue

    oil = dc['oil_exporter'].iloc[0]
    row = {'country': ctry, 'oil_exporter': oil}

    for model, dcol in [('M1', 'd_spread_m1'), ('M3', 'd_spread_m3')]:
        y = dc['d_cds'].values
        X = sm.add_constant(dc[dcol].values)
        try:
            reg = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
        except Exception:
            continue
        row[f'beta_{model}']  = reg.params[1]
        row[f'R2_{model}']    = reg.rsquared
        row[f'tstat_{model}'] = reg.tvalues[1]

    if 'R2_M1' in row and 'R2_M3' in row:
        row['R2_gain'] = row['R2_M3'] - row['R2_M1']
        cc_rows.append(row)

cc_df = pd.DataFrame(cc_rows)

print(f'\n{"Country":<20s} {"Oil":>3s} {"R2_M1":>7s} {"R2_M3":>7s} {"R2_gain":>8s} '
      f'{"beta_M1":>8s} {"beta_M3":>8s}')
print('-' * 80)
for _, r in cc_df.sort_values('oil_exporter', ascending=False).iterrows():
    tag = 'OIL' if r['oil_exporter'] == 1 else 'CTL'
    marker = ' <--' if r['R2_gain'] > 0 else ''
    print(f'{r["country"]:<20s} [{tag}] {r["R2_M1"]:>7.4f} {r["R2_M3"]:>7.4f} '
          f'{r["R2_gain"]:>+8.4f} {r["beta_M1"]:>8.3f} {r["beta_M3"]:>8.3f}{marker}')

# ---- DiD on R2 gain ----
print('\n' + '-' * 80)
for grp, label in [(1, 'Oil Exporters'), (0, 'Controls')]:
    sub = cc_df[cc_df['oil_exporter'] == grp]
    if len(sub) == 0: continue
    pct_pos = (sub['R2_gain'] > 0).mean() * 100
    print(f'  {label} (n={len(sub)}):  '
          f'avg R2_gain = {sub["R2_gain"].mean():+.4f}  '
          f'positive: {pct_pos:.0f}%')

oil_gain  = cc_df[cc_df['oil_exporter']==1]['R2_gain'].mean()
ctrl_gain = cc_df[cc_df['oil_exporter']==0]['R2_gain'].mean()
print(f'\n  DiD: Oil avg gain ({oil_gain:+.4f}) - Control avg gain ({ctrl_gain:+.4f}) '
      f'= {oil_gain - ctrl_gain:+.4f}')
print(f'  {"M3 helps oil exporters MORE" if oil_gain > ctrl_gain else "No differential"}')


In [ ]:
# ================================================================
# CONDITIONAL ACCURACY: STRESS vs CALM
# ================================================================
#
# The structural argument: M3 should help most when OVX is high
# (stress regime), because that's when jumps fire in Phase 1.
# During calm, M3 ≈ M1 (few jumps), so no improvement expected.
#
# Split sample at OVX 75th percentile, run change-on-change
# in each regime, compare R2 gains.
# ================================================================

ovx_p75 = df_res['OVX'].quantile(0.75)
df_res_sorted['ovx_stress'] = (df_res_sorted['OVX'] >= ovx_p75).astype(int)

print(f'OVX stress threshold (p75): {ovx_p75:.1f}')
print(f'Stress weeks: {df_res_sorted["ovx_stress"].sum()} / {len(df_res_sorted)}\n')

for regime, rlabel in [(0, 'CALM (OVX < p75)'), (1, 'STRESS (OVX >= p75)')]:
    print('=' * 80)
    print(f'  {rlabel}')
    print('=' * 80)

    regime_rows = []
    for ctry in sorted(df_res_sorted['country_clean'].unique()):
        dc = df_res_sorted[(df_res_sorted['country_clean'] == ctry) &
                           (df_res_sorted['ovx_stress'] == regime)].dropna(
            subset=['d_cds', 'd_spread_m1', 'd_spread_m3'])
        if len(dc) < 30:
            continue

        oil = dc['oil_exporter'].iloc[0]
        row = {'country': ctry, 'oil_exporter': oil}

        for model, dcol in [('M1', 'd_spread_m1'), ('M3', 'd_spread_m3')]:
            y = dc['d_cds'].values
            X = sm.add_constant(dc[dcol].values)
            try:
                reg = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
            except Exception:
                continue
            row[f'R2_{model}'] = reg.rsquared

        if 'R2_M1' in row and 'R2_M3' in row:
            row['R2_gain'] = row['R2_M3'] - row['R2_M1']
            regime_rows.append(row)

    rdf = pd.DataFrame(regime_rows)
    for grp, label in [(1, 'Oil Exporters'), (0, 'Controls')]:
        sub = rdf[rdf['oil_exporter'] == grp]
        if len(sub) == 0: continue
        pos = (sub['R2_gain'] > 0).sum()
        print(f'  {label} (n={len(sub)}):  avg R2_gain = {sub["R2_gain"].mean():+.4f}  '
              f'positive: {pos}/{len(sub)}')
    print()

print('  KEY TEST: R2_gain should be larger in STRESS for oil exporters.')
print('  In CALM, M3 fires few jumps, so gain should be near zero.')


In [ ]:
# ================================================================
# VISUALISATION: DD GAP (M3-M1) vs OVX
# ================================================================
# For selected countries, show how the DD gap tracks OVX.
# Oil exporters should show a clear negative relationship
# (higher OVX => M3 pushes DD down relative to M1).
# ================================================================

showcase = ['colombia', 'saudi arabia', 'brazil',
            'south africa', 'turkey', 'chile']

fig, axes = plt.subplots(3, 2, figsize=(16, 13))

for ax, ctry in zip(axes.flat, showcase):
    dc = df_res[df_res['country_clean'].str.lower() == ctry].sort_values('date')
    if len(dc) == 0:
        ax.set_title(f'{ctry.title()} — no data')
        continue

    oil = dc['oil_exporter'].iloc[0]
    tag = 'OIL' if oil == 1 else 'CTRL'
    color = 'firebrick' if oil == 1 else 'steelblue'

    # DD gap on left axis
    ax.plot(dc['date'], dc['dd_diff_m3_m1'], color=color, lw=1.2,
            label='DD gap (M3-M1)')
    ax.axhline(0, color='black', lw=0.6, ls=':')
    ax.set_ylabel('DD_M3 - DD_M1', color=color)
    ax.tick_params(axis='y', labelcolor=color)

    # OVX on right axis
    ax2 = ax.twinx()
    ax2.fill_between(dc['date'], dc['OVX'], alpha=0.15, color='orange',
                     label='OVX')
    ax2.set_ylabel('OVX', color='orange')
    ax2.tick_params(axis='y', labelcolor='orange')

    # Correlation
    corr = dc[['dd_diff_m3_m1', 'OVX']].corr().iloc[0, 1]
    ax.set_title(f'{ctry.title()} [{tag}]  corr(gap, OVX) = {corr:.3f}')
    ax.grid(True, alpha=0.2)

plt.suptitle('DD Gap (M3-M1) vs OVX Level\n'
             'Negative gap = M3 sees more risk. Should correlate with OVX for oil exporters.',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

# Summary: correlation DD_gap vs OVX by group
print('\nCorrelation: DD_gap(M3-M1) vs OVX')
print('-' * 45)
for ctry in sorted(df_res['country_clean'].unique()):
    dc = df_res[df_res['country_clean'] == ctry]
    oil = dc['oil_exporter'].iloc[0]
    corr = dc[['dd_diff_m3_m1', 'OVX']].corr().iloc[0, 1]
    tag = 'OIL' if oil == 1 else 'CTL'
    print(f'  {ctry:<20s} [{tag}]  corr = {corr:+.3f}')


In [ ]:
# ================================================================
# CROSS-COUNTRY COMPARISON
# ================================================================

rows = []
for ctry, grp in df_res.groupby('country_clean'):
    obs = grp['cds_spread']
    oil = grp['oil_exporter'].iloc[0] if 'oil_exporter' in grp.columns else np.nan
    row = {'country': ctry, 'oil_exporter': oil, 'n_obs': len(grp),
           'avg_cds': obs.mean(), 'avg_scale': grp['scale_mult'].mean()}

    for m, col in [('m1','spread_m1'),('m2','spread_m2'),('m3','spread_m3')]:
        impl = grp[col]
        row[f'rmse_{m}'] = np.sqrt(np.mean((obs - impl)**2))
        row[f'corr_{m}'] = obs.corr(impl)
    rows.append(row)

summary = pd.DataFrame(rows).sort_values('oil_exporter', ascending=False)

# Improvement: positive means M3 is better
summary['rmse_improvement_m3_vs_m1'] = summary['rmse_m1'] - summary['rmse_m3']
summary['rmse_improvement_m3_vs_m2'] = summary['rmse_m2'] - summary['rmse_m3']

print('Cross-Country Model Comparison')
print('=' * 90)
display_cols = ['country', 'oil_exporter', 'avg_cds', 'avg_scale',
                'rmse_m1', 'rmse_m2', 'rmse_m3',
                'rmse_improvement_m3_vs_m1', 'rmse_improvement_m3_vs_m2']
print(summary[display_cols].to_string(index=False, float_format='%.1f'))

# Group averages
if 'oil_exporter' in summary.columns:
    print('\nGroup Averages:')
    group_avg = summary.groupby('oil_exporter')[[
        'rmse_m1', 'rmse_m2', 'rmse_m3',
        'rmse_improvement_m3_vs_m1', 'rmse_improvement_m3_vs_m2'
    ]].mean()
    print(group_avg.round(1).to_string())